# 🚀 SVG-IR: Spatially-Varying Gaussian Splatting for Inverse Rendering
### Google Colab Notebook for **TensoIR Lego** Dataset Training & Evaluation

**Features & Workflow:**
1. 🔗 **Mount Google Drive** right at the beginning.
2. 📦 **Clone SVG-IR & Install Dependencies** (including CUDA rasterizer extensions: `bvh`, `simple-knn`, `rgss-rasterization`, `svgss_rasterization`).
3. 📥 **Download TensoIR Lego dataset** automatically from Zenodo.
4. 🏋️ **Run Stage 1 & Stage 2 Training + Evaluation**.
5. 💾 **Save all results to Google Drive** (`/content/drive/MyDrive/SVG_IR_results/TensoIR_lego`).
6. 🔌 **Disconnect Colab Runtime** automatically when finished.

In [ ]:
# ==========================================
# STEP 1: MOUNT GOOGLE DRIVE IMMEDIATELY
# ==========================================
from google.colab import drive
import os

print("Connecting to Google Drive...")
drive.mount('/content/drive')

# Destination folder on Google Drive for saving outputs
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/SVG_IR_results/TensoIR_lego'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"\n Google Drive mounted! Outputs will be saved to:\n  {DRIVE_OUTPUT_DIR}")

In [ ]:
# ==========================================
# STEP 2: CLONE REPO & INSTALL DEPENDENCIES
# ==========================================
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

# Clean old clone if re-running
!rm -rf /content/SVG-IR

# Clone repository
!git clone --recursive https://github.com/learner-shx/SVG-IR.git /content/SVG-IR
%cd /content/SVG-IR

# Install python dependencies and build tools
!pip install wheel setuptools kornia==0.6.12 plyfile trimesh imageio scikit-image opencv-python tqdm lpips
!pip install slangtorch

# Install nvdiffrast
!pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git || pip install nvdiffrast

# Install PyTorch Scatter
!pip install --no-build-isolation torch-scatter

# Build and install CUDA submodules using setup.py directly
print("\nCompiling CUDA extensions and submodules...")
!cd /content/SVG-IR/submodules/bvh && python setup.py install
!cd /content/SVG-IR/submodules/simple-knn && python setup.py install
!cd /content/SVG-IR/rgss-rasterization && python setup.py install
!cd /content/SVG-IR/svgss_rasterization && python setup.py install

print("\n Environment setup & compilation finished successfully!")

In [ ]:
# ==========================================
# STEP 3: DOWNLOAD & EXTRACT TENSOIR LEGO DATASET
# ==========================================
%cd /content/SVG-IR
!mkdir -p dataset/TensoIR

# Download TensoIR Lego scene from Zenodo (record 7880113)
print("Downloading TensoIR Lego dataset from Zenodo...")
!wget -O dataset/TensoIR/lego.zip https://zenodo.org/records/7880113/files/lego.zip

# Unzip dataset
print("Extracting dataset...")
!unzip -q dataset/TensoIR/lego.zip -d dataset/TensoIR/
!rm dataset/TensoIR/lego.zip

print("\n Dataset extracted! Directory structure:")
!ls -la dataset/TensoIR/lego

In [ ]:
# ==========================================
# STEP 4: TRAIN & EVALUATE SVG-IR (STAGE 1 & STAGE 2)
# ==========================================
%cd /content/SVG-IR

# ------------------------------------------
# STAGE 1: 3DGS Geometry Training & Evaluation
# ------------------------------------------
print("\n==========================================")
print("STARTING STAGE 1: Geometry Optimization (30,000 iterations)")
print("==========================================\n")
!python train.py --eval \
    -s dataset/TensoIR/lego \
    -m output/TensoIR/lego/gss \
    --lambda_normal_render_depth 0.0 \
    --lambda_normal_smooth 0.02 \
    --lambda_mask_entropy 0.1 \
    --save_training_vis \
    --densify_grad_normal_threshold 1e-8 \
    --lambda_depth_var 1e-2

print("\n==========================================")
print("STAGE 1 EVALUATION")
print("==========================================\n")
!python eval_nvs.py --eval \
    -m output/TensoIR/lego/gss \
    -c output/TensoIR/lego/gss/chkpnt30000.pth

# ------------------------------------------
# STAGE 2: PBR Inverse Rendering Training & Evaluation
# ------------------------------------------
print("\n==========================================")
print("STARTING STAGE 2: PBR Inverse Rendering (50,000 iterations)")
print("==========================================\n")
!python train.py --eval \
    -s dataset/TensoIR/lego \
    -m output/TensoIR/lego/render_relight \
    -c output/TensoIR/lego/gss/chkpnt30000.pth \
    --save_training_vis \
    --position_lr_init 0.00000 \
    --position_lr_final 0.0 \
    --normal_lr 0.001 \
    --sh_lr 0.00025 \
    --opacity_lr 0.005 \
    --scaling_lr 0.0000 \
    --rotation_lr 0.000 \
    --iterations 50000 \
    --lambda_base_color_smooth 0.1 \
    --lambda_roughness_smooth 0.05 \
    --lambda_light_smooth 0.0 \
    --lambda_light 0.00 \
    -t render_relight --sample_num 64 \
    --save_training_vis_iteration 200 \
    --lambda_env_smooth 0.02 \
    --env_resolution 32

print("\n==========================================")
print("STAGE 2 EVALUATION")
print("==========================================\n")
!python eval_nvs.py --eval \
    -m "output/TensoIR/lego/render_relight" \
    -c "output/TensoIR/lego/render_relight/chkpnt50000.pth" \
    -t render_relight \
    --skip_train

print("\n Training & Evaluation completed!")

In [ ]:
# ==========================================
# STEP 5: SAVE RESULTS TO GOOGLE DRIVE & DISCONNECT
# ==========================================
from google.colab import runtime
import os

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/SVG_IR_results/TensoIR_lego'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print("Copying output files to Google Drive...")
!cp -r /content/SVG-IR/output/TensoIR/lego/* "/content/drive/MyDrive/SVG_IR_results/TensoIR_lego/"

print(f"\n All training checkpoints and evaluation outputs saved to:\n  /content/drive/MyDrive/SVG_IR_results/TensoIR_lego/")

print("\nDisconnecting Google Colab runtime now to avoid unneeded compute usage...")
runtime.unassign()